# Blank / mismatched-image control

Tests whether the fine-tuned model is actually using the image, or answering
from question phrasing and learned answer-distribution priors alone. Two
conditions, both run against the **fine-tuned** model with the same questions
as the real eval, but with the image replaced:

- **Blank**: a uniform gray image carrying no scene information
- **Mismatched**: a real image, but from a different, unrelated test row (a
  derangement of the row order, so no row ever keeps its own image)

Both save predictions in the same schema as `visionllm_predictions.csv`, so
either can be dropped directly into the existing metrics notebook
(`9_evaluateVLLM_fixed.ipynb`) by pointing `RESULTS_PATH` at the corresponding
output file here. Comparing those metrics against the real-image fine-tuned
results is the actual control: if accuracy holds up close to the real-image
numbers, that\'s evidence the model isn\'t grounding its answers in the image.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

with open("/content/drive/MyDrive/Surgical-VLM/configs/config.json") as f:
    CONFIG = json.load(f)

PROCESSED_DIR = Path(CONFIG["processed_dir"])  # adjust key name if different in your config.json
CHECKPOINT = "/content/drive/MyDrive/Surgical-VLM/checkpoints/best_qwen_vl_final"

eval_df = pd.read_parquet(PROCESSED_DIR / "matched_test.parquet")
print(eval_df.shape)
eval_df.head()


(5600, 18)


,sample_id,image_id,case_id,image_path,image_full_path,image_exists,source_dataset,specialty,surgery_type,question,thinking,answer,task,task_group,gt_label,annotator,text,label_set
0,517834,29615,VID110,CholecT50/videos/VID110/001493.png,/content/CholecT50/CholecT50/videos/VID110/001...,True,CholecT50,Hepatobiliary,Cholecystectomy,Identify the grasper's action in this surgery ...,None,The grasper is performing a retract action in ...,Action Recognition,Understanding and Reasoning,"[{""actions"": [""retract""]}]",NUS,The grasper is performing a retract action in ...,[]
1,32083,67707,VID68,CholecT50/videos/VID68/001508.png,/content/CholecT50/CholecT50/videos/VID68/0015...,True,CholecT50,Hepatobiliary,Cholecystectomy,"Given the laparoscopic cholecystectomy image, ...",This is a Level 2 task requiring identificatio...,"retract, dissect",Action Recognition,Understanding and Reasoning,"[""retract, dissect""]",SJTU,"retract, dissect",[]
2,552118,31976,VID36,CholecT50/videos/VID36/000224.png,/content/CholecT50/CholecT50/videos/VID36/0002...,True,CholecT50,Hepatobiliary,Cholecystectomy,What is the grasper doing in this surgical scene?,None,The grasper is performing a retract action in ...,Action Recognition,Understanding and Reasoning,"[{""actions"": [""retract""]}]",NUS,The grasper is performing a retract action in ...,[]
3,33687,29387,VID05,CholecT50/videos/VID05/000293.png,/content/CholecT50/CholecT50/videos/VID05/0002...,True,CholecT50,Hepatobiliary,Cholecystectomy,"Given the laparoscopic cholecystectomy image, ...",This is a Level 2 task requiring identificatio...,"retract, dissect",Action Recognition,Understanding and Reasoning,"[""retract, dissect""]",SJTU,"retract, dissect",[]
4,41279,31695,VID36,CholecT50/videos/VID36/001288.png,/content/CholecT50/CholecT50/videos/VID36/0012...,True,CholecT50,Hepatobiliary,Cholecystectomy,"Given the laparoscopic cholecystectomy image, ...",This is a Level 2 task requiring identificatio...,dissect,Action Recognition,Understanding and Reasoning,"[""dissect""]",SJTU,dissect,[]


In [ ]:
import time, shutil, subprocess
from pathlib import Path

CONFIG = Path("/content/drive/MyDrive/Surgical-VLM/configs/config.json")

with open(CONFIG) as f:
    config = json.load(f)

DRIVE_CHOLECT50_ARCHIVE = Path("/content/drive/MyDrive/Surgical-VLM/data/CholecT50_raw.zip")  # adjust extension if .tar.gz
LOCAL_ARCHIVE_COPY = Path("/content/CholecT50_raw" + DRIVE_CHOLECT50_ARCHIVE.suffix)
LOCAL_CHOLECT50_DIR = Path("/content/CholecT50")

assert DRIVE_CHOLECT50_ARCHIVE.exists(), (
    f"Expected archive not found at {DRIVE_CHOLECT50_ARCHIVE}. "
    "Upload it to Drive first (see markdown above) before running this cell."
)


print("Copying archive to local disk..")
t0 = time.time()
shutil.copy2(DRIVE_CHOLECT50_ARCHIVE, LOCAL_ARCHIVE_COPY)
print(f"Copy done in {time.time()-t0:.1f}s")

LOCAL_CHOLECT50_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting locally...")
t0 = time.time()
if LOCAL_ARCHIVE_COPY.suffix == ".zip":
    subprocess.run(["unzip", "-q", str(LOCAL_ARCHIVE_COPY), "-d", str(LOCAL_CHOLECT50_DIR)], check=True)
else:
    subprocess.run(["tar", "-xzf", str(LOCAL_ARCHIVE_COPY), "-C", str(LOCAL_CHOLECT50_DIR)], check=True)
print(f"Extract done in {time.time()-t0:.1f}s")

n_files = sum(1 for _ in LOCAL_CHOLECT50_DIR.rglob("*") if _.is_file())
print(f"\n{n_files:,} files extracted to {LOCAL_CHOLECT50_DIR}")

LOCAL_ARCHIVE_COPY.unlink()



Copying archive to local disk..
Copy done in 933.6s
Extracting locally...
Extract done in 640.0s

100,918 files extracted to /content/CholecT50


## Load the fine-tuned model — merged adapter

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, CHECKPOINT)
model = model.merge_and_unload()
model.eval()

processor.tokenizer.padding_side = "left"  # required for correct batched generation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

## Prepare the two image conditions

Blank image size is taken from the first real image in the test set, so it\'s
representative of the actual input scale rather than an arbitrary fixed size.
The mismatched-image mapping is a derangement (verified below — zero rows can
keep their own image) with a fixed seed for reproducibility.

In [ ]:
from PIL import Image

# --- Blank image ---
sample_image = Image.open(eval_df.iloc[0]["image_full_path"]).convert("RGB")
BLANK_IMAGE = Image.new("RGB", sample_image.size, color=(128, 128, 128))
print("Blank image size (matches real image dimensions):", BLANK_IMAGE.size)

# --- Mismatched image mapping (derangement — no row keeps its own image) ---
def derange(n, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    while True:
        perm = rng.permutation(n)
        if not np.any(perm == idx):
            return perm

perm = derange(len(eval_df), seed=0)
mismatched_paths = eval_df["image_full_path"].values[perm]

assert not np.any(mismatched_paths == eval_df["image_full_path"].values), (
    "A row was mapped to its own image — derangement failed, check the perm logic."
)
eval_df["mismatched_image_path"] = mismatched_paths
print("Verified: 0 rows retain their own image under the mismatched mapping.")
eval_df[["image_full_path", "mismatched_image_path"]].head()


Blank image size (matches real image dimensions): (854, 480)
Verified: 0 rows retain their own image under the mismatched mapping.


,image_full_path,mismatched_image_path
0,/content/CholecT50/CholecT50/videos/VID110/001...,/content/CholecT50/CholecT50/videos/VID36/0010...
1,/content/CholecT50/CholecT50/videos/VID68/0015...,/content/CholecT50/CholecT50/videos/VID110/001...
2,/content/CholecT50/CholecT50/videos/VID36/0002...,/content/CholecT50/CholecT50/videos/VID75/0001...
3,/content/CholecT50/CholecT50/videos/VID05/0002...,/content/CholecT50/CholecT50/videos/VID36/0015...
4,/content/CholecT50/CholecT50/videos/VID36/0012...,/content/CholecT50/CholecT50/videos/VID40/0014...


## Sanity check on one sample per condition, before the full run

In [ ]:
def run_single(image, question):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": question},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    new_tokens = generated_ids[:, input_len:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

sample = eval_df.iloc[0]
mismatched_img = Image.open(sample["mismatched_image_path"]).convert("RGB")

print("Question:", sample["question"])
print("Ground Truth:", sample["answer"])
print("Prediction (blank image):     ", run_single(BLANK_IMAGE, sample["question"]))
print("Prediction (mismatched image):", run_single(mismatched_img, sample["question"]))


Question: Identify the grasper's action in this surgery image.
Ground Truth: The grasper is performing a retract action in this surgery image.
Prediction (blank image):      The grasper is performing a retract action in this surgery image.
Prediction (mismatched image): The grasper is performing a retract action in this surgery image.


## Batched generation — both conditions

In [ ]:
from tqdm.auto import tqdm

def generate_batch_control(rows, image_mode, batch_size=8):
    """image_mode: "blank" or "mismatched" """
    save_path = f"/content/drive/MyDrive/Surgical-VLM/results/visionllm_{image_mode}image_predictions.csv"
    predictions = []

    for i in tqdm(range(0, len(rows), batch_size), desc=image_mode):
        batch_rows = rows.iloc[i:i+batch_size]
        try:
            if image_mode == "blank":
                images = [BLANK_IMAGE] * len(batch_rows)
            else:
                images = [Image.open(p).convert("RGB") for p in batch_rows["mismatched_image_path"]]

            texts = []
            for img, q in zip(images, batch_rows.question):
                messages = [{"role": "user", "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": q},
                ]}]
                texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

            inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to(model.device)
            input_len = inputs["input_ids"].shape[1]

            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)

            new_tokens = generated_ids[:, input_len:]
            batch_preds = processor.batch_decode(new_tokens, skip_special_tokens=True)

        except Exception as e:
            print(f"Batch {i}-{i+batch_size} ({image_mode}) failed: {e}")
            batch_preds = [""] * len(batch_rows)

        predictions.extend(batch_preds)

        partial = rows.iloc[:len(predictions)].copy()
        partial["prediction"] = predictions
        partial[["image_full_path", "question", "answer", "task", "prediction"]].to_csv(
            save_path, index=False
        )

    return predictions, save_path


In [ ]:
blank_predictions, blank_save_path = generate_batch_control(eval_df, "blank", batch_size=8)

blank_results = eval_df.copy()
blank_results["prediction"] = blank_predictions
blank_results = blank_results[["image_full_path", "question", "answer", "task", "prediction"]]
blank_results.to_csv(blank_save_path, index=False)
print(f"Saved {len(blank_results)} blank-image predictions to {blank_save_path}")


blank:   0%|          | 0/700 [00:00<?, ?it/s]

Saved 5600 blank-image predictions to /content/drive/MyDrive/Surgical-VLM/results/visionllm_blankimage_predictions.csv


In [ ]:
mismatched_predictions, mismatched_save_path = generate_batch_control(eval_df, "mismatched", batch_size=8)

mismatched_results = eval_df.copy()
mismatched_results["prediction"] = mismatched_predictions
mismatched_results = mismatched_results[["image_full_path", "question", "answer", "task", "prediction"]]
mismatched_results.to_csv(mismatched_save_path, index=False)
print(f"Saved {len(mismatched_results)} mismatched-image predictions to {mismatched_save_path}")


mismatched:   0%|          | 0/700 [00:00<?, ?it/s]

Saved 5600 mismatched-image predictions to /content/drive/MyDrive/Surgical-VLM/results/visionllm_mismatchedimage_predictions.csv
